# 로봇팔 작업 이상 탐지와 위험성평가

149번 영상에서 사람이 표시한 이상 구간을 찾고, 힘·토크만 본 모델이 그 구간을 얼마나 잡는지 비교한다. 마지막에는 관찰한 사실과 현장에서 확인할 항목을 담은 위험성평가 초안을 저장한다.

완성 리포트의 예를 먼저 보자. 사람이 `9.63~13.57초`를 이상으로 표시했고, Random Forest는 `9.80~12.63초`에 연속 경보를 냈다. 실제 작업자의 손 접근과 방호 상태는 이 기록에서 확인할 수 없어 위험등급은 **판정 보류**로 남긴다.

예상하고 → 셀 하나 실행하고 → 출력 하나를 확인한다. 각 **코드 리뷰**에서 앞 셀을 설명한다. **파이썬 기초**와 **Q1~Q7** 셀에 직접 입력한다. 막히면 풀이 노트북에서 같은 Q 번호를 본다.

## 프로젝트 개요

| 프로젝트 단계 | 하는 일 | 단계별 산출물 |
| --- | --- | --- |
| 1. 영상과 정답 읽기 | 149번 영상을 보고 annotation.json에서 사람이 표시한 프레임을 찾는다. | 나중에 모델이 맞혔는지 비교할 **정답 구간** |
| 2. 센서 파형과 입력 만들기 | 149번 힘·토크 그래프에 정답 구간을 표시하고, 최근 약 1초씩 잘라 모델에 넣을 배열을 만든다. | **영상에서 본 변화와 센서 숫자가 연결되는 순간** |
| 3. 에피소드 분할하기 | 150개 작업을 학습 90, 검증 30, 테스트 30으로 나누고 149번이 테스트에 있는지 확인한다. | 모델이 **학습 중 보지 않은 작업**에서 잘 찾는지 평가할 조건 |
| 4. 로지스틱 회귀와 Random Forest | 1초 구간을 평균·표준편차 같은 숫자 24개로 요약해 두 분류기를 직접 학습한다. | 같은 요약값을 쓰는 **두 가지 기준 모델** |
| 5. LSTM | 센서값의 시간 순서를 살린 채 작은 배치로 학습 한 스텝을 해보고, 저장된 학습 모델을 불러온다. | **시간 순서를 쓰는 모델**이 구간마다 점수를 내는 방식 |
| 6. 탐지와 오탐 비교 | 149·8·17번에서 모델 경보와 사람의 주석을 대조한다. | **어디를 잡았고, 어디를 놓쳤는지** 보고서에 쓸 수치와 그래프 |
| 7. 현장 사전 검토 | 동일 경보에서 접근 미확인·접근 가능·차단 설계 조건을 비교한다. | **조건별 우선 확인 항목과 필요한 현장 증빙** |
| 8. 위험성평가 초안 저장 | 관찰 결과, 가정, 확인 항목, 문서 근거를 기록한다. | **현장 검증 전 위험성평가 초안** |

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
from IPython.display import Video, display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn

project = Path.cwd().resolve()
if not (project / "train.py").exists():
    project = project / "outputs" / "cria-drawer"
sys.path.insert(0, str(project))
from train import (
    CHANNELS, VALID, WRENCH, WrenchLSTM, choose_threshold, combine, evaluate,
    load_annotations, make_windows, read_episode, summarize_windows,
)

episodes = {e["episode_index"]: e for e in load_annotations()}
split = json.loads((project / "results" / "split.json").read_text(encoding="utf-8"))
print("기록 수:", len(episodes), "| Python:", sys.executable)

### 코드 리뷰 · 준비 셀

- `import`는 다른 모듈의 기능을 가져온다. 예를 들어 `Path`는 파일 경로를 다루는 도구다.
- `episodes`는 **에피소드 번호 → 주석 한 건**을 연결한 딕셔너리다. 그래서 `episodes[149]`로 149번을 바로 찾을 수 있다.
- 마지막 `sys.executable`은 지금 이 노트북 셀을 실행 중인 Python 경로다. 커널을 바꾸면 여기도 달라진다.

## 1. 149번의 영상과 정답 읽기

처음에는 사람이 이미 정답을 표시한 예제를 보고, 모델 답안을 비교할 기준을 확인한다.

1. [Hugging Face의 서랍 열기 폴더](https://huggingface.co/datasets/Joringell/CRIA/tree/main/articulated/pull_drawer)를 연다. `data`는 센서 표, `videos`는 카메라별 영상, `meta`는 파일 설명과 주석이다. 서랍 열기 에피소드 150개에 카메라가 3개라 원본 영상은 450편이다.
2. `videos → wrist → chunk-000 → episode_000149.mp4`에서 손목 영상을 확인한다. 같은 번호의 `data/chunk-000/episode_000149.parquet`는 센서 표다. 노트북 안에서 재생되지 않으면 `video`에 출력된 로컬 파일을 미디어 플레이어에서 연다.
3. `meta/annotation.json`의 원문을 열고 `"000149"`를 찾는다. `anomaly → manual → ranges`에서 `[289, 407]`을 확인한다. 이 실습의 정답은 사람이 표시한 `manual`이다. `info.json`에서 프레임 속도(30 Hz)도 찾아본다.

셀을 실행하면 파일을 연 코드가 같은 값을 꺼낸다. `episode_label`은 작업 전체의 결과(1 성공, 0 실패)이고, `frame_label`은 각 순간의 상태(0 이상, 1 정상)다. 학습용 표에서는 모델이 찾을 이상을 `1`로 바꿔 사용한다.

In [ ]:
episode_id = 149
example = read_episode(episodes[episode_id])
print(example.select("frame_index", "timestamp", "episode_label", "frame_label").head(4))
manual_ranges = episodes[episode_id]["anomaly"]["manual"]["ranges"]
times = example["timestamp"].to_numpy()
print("최종 결과:", example["episode_label"][0])
print("이상 구간(초):", [(round(float(times[a]), 2), round(float(times[b]), 2)) for a, b in manual_ranges])

video = project / "data/videos/wrist/chunk-000/episode_000149.mp4"
print("영상 파일:", video)
display(Video(filename=str(video), embed=True, width=400))

### 코드 리뷰 · 프레임 번호를 영상 시간으로 바꾸기

- `episodes[episode_id]`는 주석 한 건, `read_episode(...)`는 그 기록의 센서 표를 읽는다. `select(...).head(4)`는 필요한 열만 고르고 위에서 네 줄만 보여준다.
- `["anomaly"]["manual"]["ranges"]`는 JSON 상자 안에 있는 상자를 하나씩 여는 순서다. 149번에서는 `[[289, 407]]`이 나온다.
- `[(...) for a, b in manual_ranges]`는 시작과 끝을 한 쌍씩 꺼내는 반복문을 한 줄로 쓴 것이다. 아래에서 같은 작업을 여러 줄로 직접 해 본다.
- `times[289]`는 이 기록에서 **순번 289**에 대응하는 실제 기록 시간이다. 이상 구간 자체는 사람이 붙인 주석에서 왔다.

### 파이썬 기초 · 프레임 번호와 시간을 직접 출력

위 셀은 그대로 둔다. 아래 코드 셀에 한 줄씩 직접 입력하고 `Shift+Enter`로 실행한다.

1. **변수와 리스트 인덱스:** `frame = 289`로 프레임 번호를 변수에 담고, `print(times[frame])`으로 시간을 출력한다. 값을 `407`로 바꾸면 출력이 어떻게 바뀔지 먼저 예상한다.
2. **딕셔너리:** `print(episodes[episode_id]["anomaly"]["manual"]["ranges"])`를 입력해 사람이 주석을 단 시작과 끝을 확인한다.
3. **반복문:** `for frame in [289, 407]:`를 입력하고, 다음 줄을 네 칸 들여쓴다. 프레임 번호와 초를 함께 출력한다. 소수 둘째 자리까지 보고 싶으면 `round(float(times[frame]), 2)`를 사용한다.

확인할 출력: `289 9.63`, `407 13.57`. 그런 다음 Q1에서 센서 표의 두 행을 직접 고른다.

In [ ]:
frame = 289
print(times[frame])
frame = 407
print(times[frame])
print(episodes[episode_id]["anomaly"]["manual"]["ranges"])
for frame in [289, 407]:
    print(frame, round(float(times[frame]), 2))

**Q1. 직접 해 보기**

289번과 407번 프레임을 골라 `timestamp` 열을 읽는다. `episode_label`과 `frame_label`이 각각 무슨 뜻인지 말해 본다.

힌트는 `example.filter(pl.col("frame_index").is_in([289, 407]))`다. 확인할 출력은 첫 행 `9.63초`, 둘째 행 `13.57초`이며 두 행의 `episode_label`은 1, `frame_label`은 0이다.

In [ ]:
print(example.filter(pl.col("frame_index").is_in([289, 407])).select(
    "frame_index", "timestamp", "episode_label", "frame_label"
))

**다음 셀을 실행하기 전에 데이터 상태 확인**

아래 셀은 149번의 실제 NULL 개수와, 유효한 힘·토크 샘플이 네 개보다 적은 프레임을 센다. 149번에서는 둘 다 0이다. 전체 150개를 검사했을 때 NULL은 없었지만, 유효 샘플이 부족한 프레임은 66개였다.

In [ ]:
print("149번 열별 NULL 개수:")
print(example.null_count())
print("센서 샘플이 4개 미만인 프레임:", (example[VALID] < 4).sum())

## 2. 센서 파형을 모델 입력으로 바꾸기

30 Hz 기록 한 줄에 120 Hz의 센서 샘플이 최대 네 개 있다. 한 평가 시점까지의 **최근 30프레임**, 즉 최대 120개 센서 샘플을 묶고 5프레임마다 반복한다. 유효 샘플이 모자란 프레임이 들어간 창은 건너뛴다. 각 창의 정답은 마지막 프레임의 주석이다.

직접 해 볼 것. 아래에서 창 하나의 크기를 확인하고, 같은 창을 시간순 120×6 배열과 요약 숫자 24개로 각각 읽어 본다. 단위가 다른 힘과 토크는 그래프를 분리한다.

In [ ]:
signals = np.array([
    np.asarray(chunk[:count]).mean(axis=0) if count else np.full(6, np.nan)
    for chunk, count in zip(example[WRENCH].to_list(), example[VALID].to_list())
])

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True, constrained_layout=True)
for offset, axis in zip((0, 3), axes):
    axis.plot(times, signals[:, offset:offset + 3], label=CHANNELS[offset:offset + 3])
    for a, b in manual_ranges:
        axis.axvspan(times[a], times[b], color="tab:orange", alpha=.2)
    axis.legend(loc="upper right")
axes[0].set_ylabel("Force")
axes[1].set(ylabel="Torque", xlabel="Time (s)")
plt.show()

### 코드 리뷰 · 그리는 데이터와 학습 데이터

- `zip(...)`은 시간 순서대로 센서 데이터 묶음과 유효 샘플 수를 하나씩 짝짓는다. `chunk[:count]`는 그중 유효한 샘플만 고른다.
- `mean(axis=0)`은 한 프레임 내에서 같은 센서축끼리 평균을 내 그래프에 찍을 숫자 6개를 만든다. 학습 창은 다음 셀에서 원래 시간 순서를 유지한다.
- `axvspan`은 사람이 표시한 구간을 그래프에 칠한다. 주황색은 모델이 낸 경보가 아니다.

먼저 주황색 정답 구간의 힘과 토크를 본다. 색칠한 부분은 사람의 주석이다. 다음 셀에서는 지금 본 신호를 약 1초 분량의 모델 입력으로 묶는다.

In [ ]:
windows, labels, frame_ids, window_times = make_windows(episodes[episode_id])
print("시간순 데이터:", windows.shape, "| 요약:", summarize_windows(windows).shape)
print("첫 평가 시점(초):", round(float(window_times[0]), 3))

### 코드 리뷰 · 배열의 차원 읽기

`make_windows`는 네 값을 돌려준다. `windows`는 센서 입력, `labels`는 각 창 끝의 정답, `frame_ids`는 그 끝 프레임 번호, `window_times`는 그 시간이다. `windows[0, :, 0]`은 첫 창에서 시간 전체(`:`)를 골라 첫 번째 축(Fx)만 꺼낸다. `.shape`로 창 수·시간 샘플 수·센서축 수를 확인한다.

**Q2. 직접 해 보기**

첫 창의 여섯 센서축 중 Fx 평균을 직접 계산한다. 힌트: 창 → 시간 → 축 순서로 고른다.

확인할 출력: (120, 6), Fx 평균 한 값, (1, 24)

In [ ]:
first_fx = windows[0, :, 0]
print(windows[0].shape, round(float(first_fx.mean()), 4), summarize_windows(windows[:1]).shape)

## 3. 기록부터 분할하기

같은 영상 속 인접한 창은 센서 샘플을 상당 부분 공유한다. 창을 마구 섞으면 한 영상의 거의 같은 부분을 학습과 평가에 같이 넣게 된다. 사람 시연/정책 실행과 이상 유무를 기준으로 층화하고 **기록 단위로 90/30/30**으로 나눈다. 149·8·17번은 테스트에만 들어가 있다.

In [ ]:
names = ("train", "validation", "test")
groups = {name: set(split[name]) for name in names}
assert all(not (groups[names[i]] & groups[names[j]]) for i in range(3) for j in range(i+1, 3))
assert {8, 17, 149} <= groups["test"]
window_cache = {episode_id: make_windows(e) for episode_id, e in episodes.items()}
sets = {name: combine(window_cache, split[name]) for name in names}
for name, (x, y, _, _) in sets.items():
    print(name, "기록", len(groups[name]), "창", len(y), "이상 비율", round(y.mean(), 3))
train_x, train_y, _, _ = sets["train"]
val_x, val_y, _, _ = sets["validation"]
test_x, test_y, _, _ = sets["test"]

### 코드 리뷰 · 겹치지 않는 분할 확인

`set`은 중복 없는 모음이다. `A & B`는 양쪽에 있는 공통 원소를 구한다. `assert`는 조건이 틀리면 셀을 멈추기 때문에 같은 에피소드가 두 그룹에 들어간 채로 모델을 학습하는 실수를 막는다. `window_cache`에 에피소드별 창을 만들고, `combine`으로 분할별로 묶는다.

**Q3. 직접 해 보기**

학습과 테스트에 같은 에피소드가 들어갔는지 교집합으로 확인한다. 8·17·149번은 어디 있나?

확인할 출력: set(), True

In [ ]:
print(set(split["train"]) & set(split["test"]))
print({8, 17, 149} <= set(split["test"]))

## 4. 로지스틱 회귀와 Random Forest 직접 학습하기

이 문제는 이상/정상의 **이진 분류**다. 선형 기준으로 로지스틱 회귀를 쓴다. 로지스틱 회귀는 요약 변수에 가중치를 곱해 더한 점수를 시그모이드로 변환한다. Random Forest는 여러 결정 트리가 요약값에 서로 다른 조건을 적용하고 결과를 합친다.

두 모델에는 6축 각각의 평균, 표준편차, 최대 절댓값, 처음과 끝의 변화량을 넣는다. 두 모델의 임계값은 검증 세트에서 F1이 최대인 값으로 고르고 테스트에 적용한다. 아래 셀에서 입력과 기본 모델을 확인한다.

In [ ]:
features = {name: summarize_windows(values[0]) for name, values in sets.items()}
forest = RandomForestClassifier(
    n_estimators=100, class_weight="balanced_subsample", n_jobs=4, random_state=42
)
linear = make_pipeline(
    StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
)
for name, model in [("Logistic regression", linear), ("Random Forest", forest)]:
    model.fit(features["train"], train_y)
    val_scores = model.predict_proba(features["validation"])[:, 1]
    threshold = choose_threshold(val_y, val_scores)
    print(name, "검증 임계값", round(threshold, 3),
          "테스트", {key: round(value, 3) for key, value in
                      evaluate(test_y, model.predict_proba(features["test"])[:, 1], threshold).items()})

### 코드 리뷰 · 두 분류기를 공정하게 비교하기

- `fit`은 학습 데이터로 모델을 맞춘다. 로지스틱 회귀와 Random Forest는 동일한 요약 특성 24개를 받는다.
- `predict_proba(... )[:, 1]`은 모든 창의 이상(클래스 1) 점수만 골라낸다. `choose_threshold`는 **검증 데이터**로 경보 기준을 고른다.
- `evaluate`가 하는 일은 고른 기준으로 **테스트 데이터**의 성능을 계산하는 것이다.

**Q4. 직접 해 보기**

`features["train"]`의 행과 열이 각각 무엇인지 설명한다. 두 모델에 동일한 입력이 들어가는지 확인한다.

확인할 출력: 10116개 학습 창, 요약 특성 24개. 두 모델 모두 24개.

In [ ]:
print(features["train"].shape, len(train_y), forest.n_features_in_, linear.n_features_in_)

## 5. LSTM이 시간 순서를 읽는 방법

LSTM은 요약 전의 `[배치, 시간, 6축]`을 받는다. 마지막 은닉 상태를 선형층에 넣으면 창마다 점수 하나(로짓)가 나온다. 학습에서는 BCEWithLogitsLoss에 로짓을 넣고, 표시할 때 시그모이드를 한 번 적용한다.

첫 셀은 작은 배치에서 학습 루프의 한 번을 확인한다. 통계는 학습 데이터에서만 구한다. 두 번째 셀에서 GPU로 저장한 체크포인트를 불러와 입력과 출력 모양을 확인한다.

In [ ]:
torch.manual_seed(42)
mean = train_x.mean(axis=(0, 1))
scale = train_x.std(axis=(0, 1)).clip(min=1e-6)
batch = torch.from_numpy((train_x[:8] - mean) / scale)
targets = torch.from_numpy(train_y[:8].astype("float32"))
sample_model = WrenchLSTM()
optimizer = torch.optim.Adam(sample_model.parameters(), lr=.001)
loss_fn = nn.BCEWithLogitsLoss()
optimizer.zero_grad()
logits = sample_model(batch)
loss = loss_fn(logits, targets)
loss.backward()
optimizer.step()
print("입력:", batch.shape, "로짓:", logits.shape, "loss:", round(loss.item(), 4))

### 코드 리뷰 · LSTM 학습 한 번 따라가기

`batch`는 첫 8개 창, `targets`는 그 창들의 정답이다. `sample_model(batch)`로 예측 → `loss_fn`으로 정답과 차이 계산 → `backward`로 조정 방향 계산 → `step`으로 가중치 갱신 순서다. 잠시 후 불러올 전체 학습 모델과 여기서 한 번 갱신한 모델은 서로 별개다.

**Q5. 직접 해 보기**

LSTM에 넣는 `batch`의 차원 세 개와 logits의 역할을 말해 본다. `sigmoid`는 어느 지점에서 필요한가?

확인할 출력: 입력은 배치×시간×센서축. logits는 창당 숫자 하나. sigmoid는 점수로 읽을 때 사용.

In [ ]:
print(batch.shape, logits.shape, logits.sigmoid()[:3].detach())

In [ ]:
checkpoint = torch.load(project / "results" / "lstm.pt", map_location="cpu", weights_only=True)
trained = WrenchLSTM(checkpoint["hidden_size"])
trained.load_state_dict(checkpoint["state_dict"])
trained.eval()
with torch.inference_mode():
    sample = torch.from_numpy(windows[:8])
    score = torch.sigmoid(trained((sample - checkpoint["mean"]) / checkpoint["scale"]))
print("149번 처음 8개 점수:", score.round(decimals=3).tolist())

## 6. 모델이 맞힌 구간과 놓친 구간

AP(Average Precision)는 임계값을 하나로 고정하지 않고 점수 순위로 이상을 얼마나 잘 앞에 놓았는지 본다. Precision은 경보 중 맞은 비율, Recall은 정답 이상점 중 잡은 비율이다. 검증과 테스트에서 순위가 같았는지 확인한다.

각 점수는 **작업 이상 점수**다. 데모의 고·중·저는 작업자 노출, 가능한 피해, 방호 상태 등 가상 작업장 입력을 사용한다.

In [ ]:
report = pl.read_csv(project / "results" / "model_comparison.csv")
display(report.with_columns(pl.selectors.float().round(3)))

prediction = pl.read_csv(project / "results" / "episode_000149_scores.csv")
fig, ax = plt.subplots(figsize=(10, 3.8), constrained_layout=True)
for name, label in [("logistic_regression", "Logistic regression"), ("random_forest", "Random Forest"), ("lstm", "LSTM")]:
    ax.plot(prediction["time_s"], prediction[f"{name}_score"], label=label)
    alert = prediction.filter(pl.col(f"{name}_flag") == 1)
    ax.scatter(alert["time_s"], alert[f"{name}_score"], s=11)
for a, b in manual_ranges:
    ax.axvspan(times[a], times[b], color="tab:orange", alpha=.2, label="Annotated anomaly")
ax.set(xlabel="Time (s)", ylabel="Model score", ylim=(0, 1.03))
ax.legend(loc="upper left", ncol=2)
plt.show()

### 코드 리뷰 · 점수, 경보, 사람의 주석을 나눠 읽기

`read_csv`로 저장된 점수 표를 불러온다. `plot`은 시간별 점수를 선으로, `scatter`는 모델이 경보를 낸 지점을 점으로 표시한다. `filter(... == 1)`은 경보 행만 고른다. 경보 중 사람 주석과 겹치지 않는 창을 세면 이 비교에서의 오탐 수가 된다.

**Q6. 직접 해 보기**

저장된 149번 CSV에서 주석 창·탐지·오탐을 직접 계산한다. 같은 코드를 8·17에도 적용한다.

확인할 출력: 149번: 주석 24개, RF 18/1, LSTM 20/7. 8번은 두 모델 모두 0/0. 17번은 주석 9개, RF 3/5, LSTM 2/12.

In [ ]:
for ep in (149, 8, 17):
    frame = pl.read_csv(project / "results" / f"episode_{ep:06}_scores.csv")
    actual = frame["actual_anomaly"] == 1
    print(ep, "주석 창", int(actual.sum()))
    for detector in ("random_forest", "lstm"):
        flag = frame[detector + "_flag"] == 1
        print(detector, "탐지", int((actual & flag).sum()), "오탐", int((~actual & flag).sum()))

임계값이 달라지면 탐지와 오탐이 어떻게 바뀔지 먼저 예상해 보자. 리포트에는 검증 세트에서 미리 정한 임계값으로 얻은 테스트 결과를 사용한다.

전체 모델 학습을 다시 해 볼 때는 프로젝트 폴더에서 `python -u train.py`를 실행한다. 재실행하면 `results`의 모델과 점수 파일이 갱신된다.

## 7. 현장 사전 검토

CRIA 149번에서 경보 시각을 가져온다. 가상 작업 셀에서 서랍 주변에 손이 닿을 수 있는지 조건을 바꿔 보고, 필요한 현장 증빙을 정리한다. [SafetyDesigner 공개 가이드](https://doc.safetics.io/sfd-docs/)의 작업 정보·배치 → 위험원 파악 → 감소대책 확인 → 보고 흐름을 참고한다. 실제 현장 등급은 담당자가 확인한 접근·노출·방호 자료가 필요하다.

In [ ]:
from safety_report import SITE_SCENARIOS, prepare_site_review, read_detection, render_site_review

evidence = read_detection(149, "random_forest")
review = prepare_site_review(evidence, "unknown")
print("경보(초):", review["flagged_times_s"])
print("우선 확인:", review["first_check"])
print("실제 위험등급:", review["actual_risk"])

### 코드 리뷰 · 관찰 결과를 현장 확인으로 이어가기

`read_detection`은 저장된 모델 경보를 읽는다. `prepare_site_review(evidence, ...)`는 경보 자료와 입력한 작업장 조건을 받아 확인할 항목을 돌려준다. `review["actual_risk"]`는 그중 실제 위험등급 항목을 꺼내며, 현장 증빙이 없으면 판정 보류다.

**Q7. 조건별 확인 항목 비교**

`reachable`과 `separation_planned`를 각각 입력해 같은 경보가 어떤 현장 확인으로 이어지는지 비교한다. 두 가정이 실제 위험등급에는 어떤 영향을 주는지 확인한다.

In [ ]:
for case in ("reachable", "separation_planned"):
    review = prepare_site_review(evidence, case)
    print(review["access_assumption"], review["first_check"], review["actual_risk"])

## 8. 위험성평가 초안 저장

가정 하나를 선택해 Markdown 기록을 저장한다. 경보 시각은 모델 출력에서 가져오고, 현장에서 검증할 항목에는 아직 확인하지 않은 상태를 남긴다.

In [ ]:
scenario = "reachable"
review = prepare_site_review(evidence, scenario)
report_path = project / "results" / f"episode_000149_precheck_{scenario}.md"
report_path.write_text(render_site_review(review), encoding="utf-8")
print(report_path.relative_to(project))

### 코드 리뷰 · 리포트 파일로 남기기

`f"…{scenario}…"`는 선택한 조건을 파일명에 넣는다. `render_site_review`가 보고서 내용을 Markdown 문자열로 만들고, `write_text(..., encoding="utf-8")`가 한글을 포함한 파일을 저장한다. 마지막 `print`에서 저장 위치를 확인한다.